# INC-004 | Erro no encerramento de uma ordem

## Relato do cliente
"Um técnico recebeu uma mensagem de erro ao encerrar uma ordem. Ao consultar novamente, viu que a ordem constava como encerrada, mas o último serviço não aparecia."

## Hipótese inicial
A rotina de encerramento da ordem de serviço (`encerrar_ordem`) viola o princípio da **Atomicidade (propriedade 'A' do ACID)**. A operação divide o encerramento em duas conexões e transações de banco de dados separadas: a primeira atualiza o status da ordem para `ENCERRADA` e efetua commit imediato; a segunda transação insere o serviço final executado. Caso ocorra qualquer falha, interrupção ou indisponibilidade entre essas etapas, a primeira transação já comitada persiste no banco, deixando a ordem encerrada sem o registro do respectivo serviço.

In [ ]:
from pathlib import Path
from dotenv import dotenv_values
import psycopg
import pandas as pd
from decimal import Decimal

raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent

config = dotenv_values(raiz / "docker" / ".env")

def conectar_testes():
    return psycopg.connect(
        host="127.0.0.1",
        port=55432,
        dbname="pbl_industria_testes",
        user="aluno",
        password=config.get("POSTGRES_PASSWORD", "pipe101198")
    )

## 1. Análise do código-fonte legado

O arquivo `aplicacao/operacoes.py` (linhas 117 a 151) implementa `encerrar_ordem`:

```python
def encerrar_ordem(numero_os: str, descricao_servico: str,
                   valor_servico: Decimal, simular_falha: bool = False) -> dict:
    if valor_servico < 0:
        raise ValueError('O valor do serviço não pode ser negativo.')

    # TRANSAÇÃO 1: Atualiza status da ordem e COMITA ao sair do with
    with conectar() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                UPDATE industria.ordens_servico
                SET status = 'ENCERRADA', encerrada_em = now()
                WHERE numero = %s AND status <> 'ENCERRADA'
                RETURNING id
            """, (numero_os,))
            linha = cur.fetchone()
            ...
            ordem_id = linha[0]

    # FALHA EXTERNA OU DE REDE INTERCEPTADA AQUI:
    if simular_falha:
        raise RuntimeError('O sistema de serviços ficou temporariamente indisponível.')

    # TRANSAÇÃO 2: Segunda conexão separada para inserir o serviço
    with conectar() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO industria.servicos_ordem
                    (ordem_servico_id, descricao, valor)
                VALUES (%s, %s, %s) RETURNING id
            """, (ordem_id, descricao_servico, valor_servico))
            servico_id = cur.fetchone()[0]

    return {'ordem_id': ordem_id, 'servico_id': servico_id, 'status': 'ENCERRADA'}
```

### Problemas estruturais:
1. **Quebra de Transação Atômica**: Ao usar dois blocos `with conectar() as conn:`, o encerramento da OS e o registro do serviço tornam-se operações desarticuladas.
2. **Estado Inconsistente Irreversível**: Se a inserção do serviço falhar, a ordem já consta como `ENCERRADA`. O técnico não consegue reexecutar a rotina porque o comando possui o filtro `WHERE status <> 'ENCERRADA'`.

In [ ]:
# Simulação da falha com uma ordem aberta de testes
os_teste = "OS-2026-003"

# 1. Garantir que a OS está em estado ABERTA/EM_EXECUCAO
with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            UPDATE industria.ordens_servico
            SET status = 'EM_EXECUCAO', encerrada_em = NULL
            WHERE numero = %s;
        """, (os_teste,))
        conn.commit()

print(f"Estado inicial de {os_teste} preparado para o experimento.")

In [ ]:
# 2. Executar a rotina legada simulando a falha
from aplicacao.operacoes import encerrar_ordem

try:
    encerrar_ordem(
        numero_os=os_teste,
        descricao_servico="Calibração final de sensores",
        valor_servico=Decimal("250.00"),
        simular_falha=True
    )
except RuntimeError as erro:
    print(f"Exceção capturada com sucesso: {erro}")

In [ ]:
# 3. Conferir o estado no banco de dados após a falha
with conectar_testes() as conn:
    with conn.cursor() as cur:
        # Verificar a ordem
        cur.execute("""
            SELECT numero, status, encerrada_em
            FROM industria.ordens_servico
            WHERE numero = %s;
        """, (os_teste,))
        os_info = cur.fetchone()
        
        # Verificar se o serviço foi registrado
        cur.execute("""
            SELECT COUNT(*)
            FROM industria.servicos_ordem s
            JOIN industria.ordens_servico o ON o.id = s.ordem_servico_id
            WHERE o.numero = %s AND s.descricao = 'Calibração final de sensores';
        """, (os_teste,))
        total_servicos = cur.fetchone()[0]

print(f"Status da OS após o erro: {os_info[1]} (encerrada_em: {os_info[2]})")
print(f"Serviços encontrados com a descrição: {total_servicos}")
assert os_info[1] == 'ENCERRADA' and total_servicos == 0, "Falha não reproduzida!"
print("INCONSISTÊNCIA CONFIRMADA: A OS foi encerrada, mas o serviço foi perdido!")

## 2. Solução proposta: Transação única atômica

Para preservar a integridade, todas as operações dependentes devem ser executadas no contexto de **uma única conexão e uma única transação**. Caso ocorra qualquer falha, o PostgreSQL realiza rollback de todas as alterações.

```python
def encerrar_ordem_atomica(numero_os: str, descricao_servico: str,
                           valor_servico: Decimal, simular_falha: bool = False) -> dict:
    if valor_servico < 0:
        raise ValueError('O valor do serviço não pode ser negativo.')

    # CONEXÃO E TRANSAÇÃO ÚNICA
    with conectar() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                UPDATE industria.ordens_servico
                SET status = 'ENCERRADA', encerrada_em = now()
                WHERE numero = %s AND status <> 'ENCERRADA'
                RETURNING id
            """, (numero_os,))
            linha = cur.fetchone()
            if linha is None:
                raise ValueError('Ordem inexistente ou já encerrada.')
            ordem_id = linha[0]

            if simular_falha:
                raise RuntimeError('Simulação de falha transacional.')

            cur.execute("""
                INSERT INTO industria.servicos_ordem
                    (ordem_servico_id, descricao, valor)
                VALUES (%s, %s, %s) RETURNING id
            """, (ordem_id, descricao_servico, valor_servico))
            servico_id = cur.fetchone()[0]
            
            # Commit automático ao sair do with sem exceções

    return {'ordem_id': ordem_id, 'servico_id': servico_id, 'status': 'ENCERRADA'}
```

In [ ]:
# Teste de validação da rotina atômica
def testar_encerramento_atomico(simular_falha):
    # Restaurar status para EM_EXECUCAO
    with conectar_testes() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                UPDATE industria.ordens_servico
                SET status = 'EM_EXECUCAO', encerrada_em = NULL
                WHERE numero = %s;
            """, (os_teste,))
            conn.commit()

    try:
        with conectar_testes() as conn:
            with conn.cursor() as cur:
                cur.execute("""
                    UPDATE industria.ordens_servico
                    SET status = 'ENCERRADA', encerrada_em = now()
                    WHERE numero = %s AND status <> 'ENCERRADA'
                    RETURNING id
                """, (os_teste,))
                ordem_id = cur.fetchone()[0]

                if simular_falha:
                    raise RuntimeError("Falha provocada para testar atomicidade!")

                cur.execute("""
                    INSERT INTO industria.servicos_ordem
                        (ordem_servico_id, descricao, valor)
                    VALUES (%s, %s, %s) RETURNING id
                """, (ordem_id, "Serviço Atômico", Decimal("100.00")))
                conn.commit()
    except Exception as e:
        print(f"Erro capturado: {e}")

    # Verificar se sofreu rollback
    with conectar_testes() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT status, encerrada_em FROM industria.ordens_servico WHERE numero = %s", (os_teste,))
            status, encerrada_em = cur.fetchone()
            print(f"Status no banco após teste com falha: {status} | Encerrada em: {encerrada_em}")
            assert status == 'EM_EXECUCAO', "Rollback falhou!"
            print("Sucesso: A ordem permaneceu intacta em EM_EXECUCAO após o erro!")

testar_encerramento_atomico(simular_falha=True)

# Ficha técnica | INC-004

## 1. Hipótese e resultado esperado
O técnico relatou erro durante o encerramento da ordem, com o status mudando para encerrada mas sem o serviço cadastrado.
- **Hipótese:** Violação do princípio de atomicidade por divisão da rotina em duas conexões/transações independentes, gravando o UPDATE da OS antes da inserção do serviço.
- **Resultado esperado:** O encerramento da OS e a inclusão do serviço final devem ocorrer sob a mesma transação. Diante de qualquer erro, deve ocorrer rollback total, mantendo a OS no status anterior.

## 2. Procedimento de investigação
1. Exame do método `encerrar_ordem` em `aplicacao/operacoes.py`.
2. Identificação da segmentação em dois blocos `with conectar() as conn:`.
3. Execução da função passando `simular_falha=True`.
4. Consulta ao banco comprovando o status indevido `ENCERRADA` e a ausência do registro em `servicos_ordem`.

## 3. Evidências e resultados
- O código confirma o encerramento com commit prematuro antes do ponto de simulação de falha.
- A reprodução experimental deixou a `OS-2026-003` com status `ENCERRADA` e zero serviços vinculados.
- O teste com transação unificada provou que, sob exceção, o banco executa rollback automático e devolve a ordem para `EM_EXECUCAO`.

## 4. Conclusão
Diagnóstico confirmado com perfeição empírica e conceitual. A falha é corrigida pela unificação do escopo transacional.

## 5. Solução proposta e validação
1. Refatorar `encerrar_ordem` para utilizar um único bloco transacional `with conectar() as conn:`.
2. Validar executando encerramentos com e sem erro forçado, comprovando integridade em ambos os casos.